In [6]:
import json
import pickle
import numpy as np
import pandas as pd
import os
print(os.getcwd())

c:\Users\atom0\OneDrive\Documents\College_folder\4th\DACN_Do-Thanh-Thai\CL-TTE\playground


In [7]:
data_path = "../../data/mydata/"
train_path = os.path.join(data_path, "train.npy")
data = np.load(train_path, allow_pickle=True)
print(f"Loaded data from {train_path}, shape: {data.shape}")

Loaded data from ../../data/mydata/train.npy, shape: (900877, 6)


In [8]:
with open(os.path.join(data_path,"network_porto/porto_edges_new_simplify.pkl"), 'rb') as f:
    edgeinfo = pickle.load(f)
with open(os.path.join(data_path,"network_porto/porto_nodes_new.pkl"), 'rb') as f:
    nodeinfo = pickle.load(f)

In [4]:
print(data[0])

[1386940210620000409
 list([10497, 3860, 2554, 2556, 6897, 1945, 1910, 2547, 3889, 9651, 9654, 4074, 9658, 8588, 3941, 9661, 10295, 9671, 3939, 3872, 3935, 6604, 3833, 2520, 3823, 3832, 6592])
 4 347 790 405]


In [5]:
times = []
for d in data:
    times.append(d[-1])
print(np.mean(times), np.std(times))

616.8322812104205 333.53738287030507


In [4]:
print(type(edgeinfo), len(edgeinfo))

<class 'dict'> 10614


In [9]:
import ast

# 1. Extract all raw types from your edgeinfo values
raw_types = {v[0] for v in edgeinfo.values()}

# 2. Function to turn "['a', 'b']" or "a" into a flat list ['a', 'b']
def listify_string(val):
    if isinstance(val, str) and val.startswith("["):
        try:
            return ast.literal_eval(val)
        except:
            return [val]
    return [val]

# 3. Flatten everything into a single set of unique "atomic" road types
atomic_types = set()
for t in raw_types:
    atoms = listify_string(t)
    for a in atoms:
        atomic_types.add(a)

# 4. Construct the highway dict
# ID 0: Reserved for Padding
# ID 1: Reserved for 'unclassified' (The Catch-all)
highway = {"<PAD>": 0, "unclassified": 1}

# Add all other types starting from ID 2
current_id = 2
for t in sorted(list(atomic_types)):
    if t != "unclassified":
        highway[t] = current_id
        current_id += 1

print(f"Total Unique Atomic Types: {len(highway)}")
print(highway)

Total Unique Atomic Types: 17
{'<PAD>': 0, 'unclassified': 1, 'busway': 2, 'crossing': 3, 'living_street': 4, 'motorway': 5, 'motorway_link': 6, 'primary': 7, 'primary_link': 8, 'residential': 9, 'road': 10, 'secondary': 11, 'secondary_link': 12, 'tertiary': 13, 'tertiary_link': 14, 'trunk': 15, 'trunk_link': 16}


In [10]:
count = {road_type : 0 for road_type in highway.keys()}

In [11]:
print(edgeinfo[0])

['motorway_link', 32.3884588871153, '25503936', '4722746638']


In [12]:
for d in data:
    edge_id_list = d[1]
    for edge_id in edge_id_list:
        edge = edgeinfo[edge_id]
        road_type = edge[0]
        if road_type.startswith("["):
            try:
                types = ast.literal_eval(road_type)
                for t in types:
                    if t in count:
                        count[t] += 1
            except:
                if road_type in count:
                    count[road_type] += 1
        else:
            if road_type in count:
                count[road_type] += 1
    
print(count)

{'<PAD>': 0, 'unclassified': 4419, 'busway': 364197, 'crossing': 2429, 'living_street': 421531, 'motorway': 1142633, 'motorway_link': 594372, 'primary': 5568568, 'primary_link': 351960, 'residential': 6027140, 'road': 664, 'secondary': 21611276, 'secondary_link': 303841, 'tertiary': 6420231, 'tertiary_link': 35444, 'trunk': 53841, 'trunk_link': 38198}


In [9]:
print(len(edgeinfo))

10614


In [ ]:
print("First data sample:")
print(data[0])

First data sample:
[1386940210620000409
 list([10497, 3860, 2554, 2556, 6897, 1945, 1910, 2547, 3889, 9651, 9654, 4074, 9658, 8588, 3941, 9661, 10295, 9671, 3939, 3872, 3935, 6604, 3833, 2520, 3823, 3832, 6592])
 4 347 790 405]


In [19]:
def parse_highway_tags(raw_val, max_tags=2):
    """Converts OSM strings/lists to a fixed-size list of IDs."""
    UNCLASSIFIED_ID = highway.get('unclassified', 1)
    
    # 1. Handle string/list input
    if isinstance(raw_val, str) and raw_val.startswith("["):
        try: tags = ast.literal_eval(raw_val)
        except: tags = [raw_val]
    elif isinstance(raw_val, list):
        tags = raw_val
    else:
        tags = [raw_val]

    # 2. Map to IDs with fallback
    ids = [highway.get(t, UNCLASSIFIED_ID) for t in tags]
    
    # 3. Pad with 0 (Reserved for 'No Tag')
    while len(ids) < max_tags:
        ids.append(0)
    return ids[:max_tags]

In [22]:
id_ = parse_highway_tags(edgeinfo[0][0])
print("Raw highway tag for edge 0:", edgeinfo[0][0])
print(f"Parsed IDs for edge 0: {id_}")

Raw highway tag for edge 0: motorway_link
Parsed IDs for edge 0: [6, 0]


In [ ]:
print("STD: ", np.std(data[:, -1]))

STD:  333.537382870301


In [ ]:
os.listdir('../')